## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-13-08-30-00 +0000,wsj,European Stock Nudge Lower in Muted Opening Trade,https://www.wsj.com/finance/stocks/nikkei-rise...
1,2026-11-13-08-21-46 +0000,nypost,Stephen A Smith debriefs interview with Presid...,https://nypost.com/2026/11/13/us-news/stephen-...
2,2026-11-13-08-20-28 +0000,bbc,Australian cricket's sandpapergate is back - a...,https://www.bbc.co.uk/news/articles/c6702z750z...
3,2026-11-13-08-16-30 +0000,nypost,"Microsoft platforms including Outlook, Teams d...",https://nypost.com/2026/11/13/business/microso...
4,2026-11-13-08-13-44 +0000,bbc,Asylum centre on military base near Piddington...,https://www.bbc.co.uk/news/articles/c58j479887...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2677/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
343,huang,44
342,jensen,39
150,nvidia,38
93,new,28
11,trump,26


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
316,2026-11-12-14-37-00 +0000,wsj,"From the outset, some Trump administration off...",https://www.wsj.com/politics/national-security...,199
42,2026-11-13-03-57-01 +0000,bbc,Ex-Nvidia CEO Jensen Huang summoned to testify...,https://www.bbc.co.uk/news/articles/cmq5vd1wdg...,185
302,2026-11-12-15-50-09 +0000,nypost,Trump floats ex-Apple CEO Tim Cook to succeed ...,https://nypost.com/2026/11/12/us-news/trump-fl...,184
331,2026-11-12-13-59-07 +0000,chitri,Nvidia CEO Jensen Huang resigns after his affa...,https://www.chicagotribune.com/2026/11/12/nvid...,166
231,2026-11-12-19-09-50 +0000,nypost,California Dem’s wife caught in curious Nvidia...,https://nypost.com/2026/11/12/us-news/rep-ro-k...,163


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
316,199,2026-11-12-14-37-00 +0000,wsj,"From the outset, some Trump administration off...",https://www.wsj.com/politics/national-security...
115,110,2026-11-12-23-05-02 +0000,missionlocal,"Richmond Buzz: New brunch, new pizza, and a ne...",https://missionlocal.org/2026/11/richmond-buzz...
307,100,2026-11-12-15-15-00 +0000,wsj,President Trump said the U.S. doesn’t need a d...,https://www.wsj.com/politics/policy/trump-now-...
47,60,2026-11-13-03-27-00 +0000,wsj,"Sen. Chris Van Hollen (D., Md.) said he would ...",https://www.wsj.com/politics/policy/chris-van-...
250,58,2026-11-12-18-17-00 +0000,wsj,Pentagon chief Pete Hegseth pinned a medal on ...,https://www.wsj.com/politics/hegseth-and-aides...
358,54,2026-11-12-11-54-00 +0000,wsj,"Welcome to the lame-duck Congress of 2026, whe...",https://www.wsj.com/politics/policy/disorganiz...
282,50,2026-11-12-17-08-00 +0000,wsj,Home sales declined in October to their lowest...,https://www.wsj.com/economy/housing/home-sales...
294,50,2026-11-12-16-29-00 +0000,wsj,"To win control of the House and Senate, Democr...",https://www.wsj.com/politics/elections/the-vot...
58,46,2026-11-13-02-45-20 +0000,bbc,'How does a building just fall?' Neighbours gr...,https://www.bbc.co.uk/news/articles/cm4gn5qddq...
65,44,2026-11-13-02-01-02 +0000,nypost,"Muslim student, 14, goes to war with Californi...",https://nypost.com/2026/11/12/us-news/muslim-s...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
